# CFD Geometry — quick start

Try **CFDGeometry** in Google Colab or locally in **VS Code / Cursor**.

| Step | Action |
|------|--------|
| 1 | Install (first code cell) |
| 2 | Draw study extent on the map |
| 3 | Build geometry (download + STL) |
| 4 | 3D preview (Plotly) |

Keep the drawn box **small** (city-block scale) so download and extrusion finish quickly.


In [ ]:
# 1) Install — run this cell first
import subprocess
import sys
from pathlib import Path

_GIT = "git+https://github.com/Omokayode/CFDGeometry.git@main"
_DEPS = (
    "ipywidgets>=7.6,<9",
    "ipyleaflet>=0.17",
    "jupyterlab_widgets>=1.0.5,<4",
    "plotly>=5.18",
)


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


repo = Path.cwd()
if _in_colab():
    _pip(*_DEPS, "--upgrade-strategy", "only-if-needed")
    _pip(
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        f"{_GIT}#egg=cfd-geometry[download]",
    )
elif (repo / "src" / "cfd_geometry").exists():
    _pip("-e", f"{repo}[notebook,download]")
else:
    _pip(f"{_GIT}#egg=cfd-geometry[notebook,download]")

from cfd_geometry.notebook import setup_colab_widgets

if setup_colab_widgets():
    print("Colab widget manager enabled.")

import importlib
import cfd_geometry
import cfd_geometry.notebook as _cfd_nb

importlib.reload(_cfd_nb)

print("cfd_geometry", cfd_geometry.__version__)


In [ ]:
# 2) Draw study extent — rectangle tool → "Use this extent"
PLACE = "Milwaukee, Wisconsin, USA"  # or CENTER = (lat, lon)
CENTER = None

from cfd_geometry.notebook import select_extent

selector = select_extent(place=PLACE if CENTER is None else None, center=CENTER)
selector


In [ ]:
if selector.bbox is None:
    raise RuntimeError("Draw a rectangle, then click 'Use this extent'.")

bbox = selector.bbox
print(
    f"west={bbox.west:.6f} south={bbox.south:.6f} "
    f"east={bbox.east:.6f} north={bbox.north:.6f}"
)


In [ ]:
# 3) Download OSM + extrude STLs
from pathlib import Path

from cfd_geometry.domain import DomainConfig, build_domain

config = DomainConfig(
    output_dir=Path("data"),
    bbox=selector.bbox,
    run_download=True,
    download_layers=("buildings", "trees"),
    download_dem=False,
    build_buildings=True,
    build_trees=True,
    build_highways=False,
    build_terrain=False,
    height_source="composite",
)

result = build_domain(config)
result.stl_files


In [ ]:
# Output files
from pathlib import Path

out = Path("data/output")
for p in sorted(out.glob("*.stl")):
    print(p.name, f"{p.stat().st_size / 1024:.1f} KiB")


In [ ]:
# 4) Interactive 3D preview (Plotly)
from cfd_geometry.notebook.visualize import plot_domain_stls

plot_domain_stls(result, layers=("buildings", "trees"), max_triangles=8000)
